# OpenAlex Final Dataset Analysis - Sri Lanka Only

This notebook loads the final OpenAlex dataset, enforces a strict Sri Lanka-only affiliation filter, and produces summary tables for publications, citations, fields, institutions, authors, journals, and open access.

**Strict Sri Lanka-only rule used here:** keep a work only when the known affiliation countries are exactly `LK`. Records with no detectable country code or with mixed countries such as `LK; US` are excluded from the strict dataset.

In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

IS_KAGGLE = Path("/kaggle/input").exists()

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
KAGGLE_INPUT_DIR = Path("/kaggle/input")

if IS_KAGGLE:
    DATA_SEARCH_DIRS = [KAGGLE_INPUT_DIR, DATA_DIR]
    OUTPUT_DIR = Path("/kaggle/working/openalex_outputs")
else:
    DATA_SEARCH_DIRS = [DATA_DIR]
    OUTPUT_DIR = DATA_DIR / "processed" / "openalex"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Running on Kaggle: {IS_KAGGLE}")
print(f"Output directory: {OUTPUT_DIR}")
PROJECT_ROOT

## 1. Select the final OpenAlex dataset

On Kaggle, add your OpenAlex final dataset using **Add Input**. This notebook will search inside `/kaggle/input` automatically.

If the wrong file is selected, manually set `DATASET_PATH`, for example:

```python
DATASET_PATH = Path("/kaggle/input/your-dataset-name/openalex_final.csv")
```

Supported formats: `.csv`, `.parquet`, `.jsonl`, `.json`, `.xlsx`.

In [ ]:
candidate_paths = []
for base_dir in DATA_SEARCH_DIRS:
    candidate_paths.extend(
        [
            base_dir / "processed" / "openalex" / "openalex_final.csv",
            base_dir / "processed" / "openalex_final.csv",
            base_dir / "interim" / "openalex" / "openalex_final.csv",
            base_dir / "raw" / "openalex" / "lk_works.csv",
            base_dir / "raw" / "openalex" / "lk_works.jsonl",
            base_dir / "openalex_final.csv",
            base_dir / "lk_works.csv",
            base_dir / "lk_works.jsonl",
        ]
    )

existing_candidates = [path for path in candidate_paths if path.exists()]
if existing_candidates:
    DATASET_PATH = existing_candidates[0]
else:
    discovered = []
    for base_dir in DATA_SEARCH_DIRS:
        if base_dir.exists():
            discovered.extend(list(base_dir.rglob("*openalex*")))
            discovered.extend(list(base_dir.rglob("*lk_works*")))
    discovered = sorted(
        {
            path.resolve()
            for path in discovered
            if path.is_file() and path.suffix.lower() in {".csv", ".parquet", ".jsonl", ".json", ".xlsx"}
        },
        key=lambda path: str(path),
    )
    if not discovered:
        raise FileNotFoundError(
            "No OpenAlex dataset found. On Kaggle, add the dataset with Add Input, then set DATASET_PATH manually if needed."
        )
    DATASET_PATH = discovered[0]

DATASET_PATH

In [ ]:
def load_dataset(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix == ".jsonl":
        return pd.read_json(path, lines=True)
    if suffix == ".json":
        return pd.read_json(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported dataset format: {path.suffix}")


df_raw = load_dataset(DATASET_PATH)
df_raw.columns = [str(column).strip() for column in df_raw.columns]

print(f"Loaded: {DATASET_PATH}")
print(f"Rows: {len(df_raw):,}")
print(f"Columns: {df_raw.shape[1]:,}")
df_raw.head(3)

## 2. Normalize important columns

This section handles both a flat CSV exported by `scripts/convert_openalex_jsonl_to_csv.py` and raw OpenAlex JSON/JSONL records.

In [ ]:
def first_existing_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    lowered = {column.lower(): column for column in df.columns}
    for candidate in candidates:
        if candidate.lower() in lowered:
            return lowered[candidate.lower()]
    return None


def normalize_text(value) -> str:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    return str(value).strip()


def split_multi_value(value) -> list[str]:
    text = normalize_text(value)
    if not text or text.lower() in {"nan", "none", "null"}:
        return []
    parts = re.split(r"\s*;\s*|\s*,\s*|\s*\|\s*", text)
    return [part.strip() for part in parts if part.strip()]


def unique_join(values, separator="; ") -> str:
    seen = set()
    cleaned = []
    for value in values:
        text = normalize_text(value)
        if not text or text in seen:
            continue
        seen.add(text)
        cleaned.append(text)
    return separator.join(cleaned)


def as_list(value) -> list:
    return value if isinstance(value, list) else []


def raw_country_codes(row: pd.Series) -> set[str]:
    codes: set[str] = set()

    countries_col = first_existing_column(df_raw, ["countries", "country_codes", "affiliation_countries"])
    if countries_col:
        codes.update(code.upper() for code in split_multi_value(row.get(countries_col)))

    authorships = row.get("authorships")
    if isinstance(authorships, str):
        try:
            authorships = json.loads(authorships)
        except json.JSONDecodeError:
            authorships = []

    for authorship in as_list(authorships):
        if not isinstance(authorship, dict):
            continue
        codes.update(str(code).upper() for code in as_list(authorship.get("countries")) if code)
        for institution in as_list(authorship.get("institutions")):
            if isinstance(institution, dict) and institution.get("country_code"):
                codes.add(str(institution["country_code"]).upper())

    institutions = row.get("institutions")
    if isinstance(institutions, str) and institutions.lstrip().startswith("["):
        try:
            institutions = json.loads(institutions)
        except json.JSONDecodeError:
            institutions = []

    for institution in as_list(institutions):
        if isinstance(institution, dict) and institution.get("country_code"):
            codes.add(str(institution["country_code"]).upper())

    return {code for code in codes if code and code != "NAN"}


def extract_raw_names(row: pd.Series, key: str, lk_only: bool = False) -> str:
    names = []
    authorships = row.get("authorships")
    if isinstance(authorships, str):
        try:
            authorships = json.loads(authorships)
        except json.JSONDecodeError:
            authorships = []

    for authorship in as_list(authorships):
        if not isinstance(authorship, dict):
            continue
        countries = {str(code).upper() for code in as_list(authorship.get("countries"))}
        institutions = as_list(authorship.get("institutions"))
        has_lk_institution = any(
            isinstance(inst, dict) and inst.get("country_code") == "LK" for inst in institutions
        )
        if lk_only and "LK" not in countries and not has_lk_institution:
            continue

        if key == "authors":
            author = authorship.get("author") or {}
            names.append(author.get("display_name") or authorship.get("raw_author_name"))
        elif key == "institutions":
            names.extend(inst.get("display_name") for inst in institutions if isinstance(inst, dict))

    return unique_join(names)


df = df_raw.copy()

rename_map = {
    first_existing_column(df, ["id", "openalex_id"]): "openalex_id",
    first_existing_column(df, ["display_name", "title"]): "title",
    first_existing_column(df, ["publication_year", "year"]): "publication_year",
    first_existing_column(df, ["publication_date", "date"]): "publication_date",
    first_existing_column(df, ["cited_by_count", "citations"]): "cited_by_count",
}
rename_map = {old: new for old, new in rename_map.items() if old and old != new}
df = df.rename(columns=rename_map)

if "affiliation_country_codes" not in df.columns:
    df["affiliation_country_codes"] = df_raw.apply(raw_country_codes, axis=1)
else:
    df["affiliation_country_codes"] = df["affiliation_country_codes"].apply(
        lambda value: value if isinstance(value, set) else {code.upper() for code in split_multi_value(value)}
    )

for column in ["publication_year", "cited_by_count", "author_count", "fwci"]:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

if "authors" not in df.columns and "authorships" in df_raw.columns:
    df["authors"] = df_raw.apply(lambda row: extract_raw_names(row, "authors"), axis=1)

if "sri_lankan_authors" not in df.columns and "authorships" in df_raw.columns:
    df["sri_lankan_authors"] = df_raw.apply(lambda row: extract_raw_names(row, "authors", lk_only=True), axis=1)

if "sri_lankan_institutions" not in df.columns and "authorships" in df_raw.columns:
    df["sri_lankan_institutions"] = df_raw.apply(lambda row: extract_raw_names(row, "institutions", lk_only=True), axis=1)

df[["openalex_id", "title", "publication_year", "cited_by_count", "affiliation_country_codes"]].head(5)

## 3. Apply strict Sri Lanka-only filter

In [ ]:
STRICT_COUNTRY_CODES = {"LK"}

df["is_strict_sri_lanka_only"] = df["affiliation_country_codes"].apply(lambda codes: codes == STRICT_COUNTRY_CODES)
df["has_any_sri_lanka_affiliation"] = df["affiliation_country_codes"].apply(lambda codes: "LK" in codes)

lk_only = df.loc[df["is_strict_sri_lanka_only"]].copy()
lk_mixed_or_unknown = df.loc[~df["is_strict_sri_lanka_only"]].copy()

summary_filter = pd.DataFrame(
    {
        "group": ["raw records", "strict Sri Lanka-only", "has LK but mixed countries", "no detectable LK / unknown"],
        "records": [
            len(df),
            len(lk_only),
            int(((df["has_any_sri_lanka_affiliation"]) & (~df["is_strict_sri_lanka_only"])).sum()),
            int((~df["has_any_sri_lanka_affiliation"]).sum()),
        ],
    }
)

display(summary_filter)

if lk_only.empty:
    raise ValueError(
        "Strict Sri Lanka-only dataset is empty. Check the country columns or use the converter with --sri-lanka-filter only."
    )

print("Unique country-code sets kept in strict dataset:")
display(lk_only["affiliation_country_codes"].value_counts())

In [ ]:
output_csv = OUTPUT_DIR / "openalex_sri_lanka_only.csv"
output_parquet = OUTPUT_DIR / "openalex_sri_lanka_only.parquet"

lk_export = lk_only.copy()
lk_export["affiliation_country_codes"] = lk_export["affiliation_country_codes"].apply(lambda codes: "; ".join(sorted(codes)))
lk_export.to_csv(output_csv, index=False)

try:
    lk_export.to_parquet(output_parquet, index=False)
    print(f"Saved CSV: {output_csv}")
    print(f"Saved Parquet: {output_parquet}")
except Exception as error:
    print(f"Saved CSV: {output_csv}")
    print(f"Skipped Parquet export because the engine is unavailable: {error}")

## 4. Dataset quality checks

In [ ]:
quality_rows = []
for column in ["openalex_id", "doi", "title", "publication_year", "publication_date", "cited_by_count", "sri_lankan_institutions"]:
    if column in lk_only.columns:
        quality_rows.append(
            {
                "column": column,
                "missing": int(lk_only[column].isna().sum() + (lk_only[column].astype(str).str.strip() == "").sum()),
                "missing_pct": round(float((lk_only[column].isna() | (lk_only[column].astype(str).str.strip() == "")).mean() * 100), 2),
                "unique": int(lk_only[column].nunique(dropna=True)),
            }
        )

display(pd.DataFrame(quality_rows))

if "openalex_id" in lk_only.columns:
    duplicate_openalex = lk_only[lk_only["openalex_id"].duplicated(keep=False)].sort_values("openalex_id")
    print(f"Duplicate OpenAlex IDs: {duplicate_openalex['openalex_id'].nunique():,}")
    display(duplicate_openalex[["openalex_id", "title", "publication_year"]].head(20))

## 5. Core publication and citation summary

In [ ]:
def safe_sum(series: pd.Series) -> float:
    return float(pd.to_numeric(series, errors="coerce").fillna(0).sum())


summary = {
    "publications": len(lk_only),
    "year_min": int(lk_only["publication_year"].min()) if "publication_year" in lk_only.columns and lk_only["publication_year"].notna().any() else None,
    "year_max": int(lk_only["publication_year"].max()) if "publication_year" in lk_only.columns and lk_only["publication_year"].notna().any() else None,
    "total_citations": int(safe_sum(lk_only["cited_by_count"])) if "cited_by_count" in lk_only.columns else None,
    "mean_citations": round(float(lk_only["cited_by_count"].mean()), 2) if "cited_by_count" in lk_only.columns else None,
    "median_citations": round(float(lk_only["cited_by_count"].median()), 2) if "cited_by_count" in lk_only.columns else None,
}

pd.DataFrame([summary]).T.rename(columns={0: "value"})

In [ ]:
if "publication_year" in lk_only.columns:
    yearly = (
        lk_only.dropna(subset=["publication_year"])
        .assign(publication_year=lambda data: data["publication_year"].astype(int))
        .groupby("publication_year", as_index=False)
        .agg(
            publications=("title", "size"),
            total_citations=("cited_by_count", "sum") if "cited_by_count" in lk_only.columns else ("title", "size"),
            mean_citations=("cited_by_count", "mean") if "cited_by_count" in lk_only.columns else ("title", "size"),
        )
        .sort_values("publication_year")
    )
    yearly["mean_citations"] = yearly["mean_citations"].round(2)
    display(yearly.tail(20))
else:
    print("No publication_year column found.")

## 6. Top publications by citations

In [ ]:
top_publication_columns = [
    column for column in [
        "title",
        "publication_year",
        "cited_by_count",
        "sri_lankan_authors",
        "sri_lankan_institutions",
        "source_name",
        "doi",
        "openalex_id",
    ] if column in lk_only.columns
]

if "cited_by_count" in lk_only.columns:
    display(lk_only.sort_values("cited_by_count", ascending=False)[top_publication_columns].head(25))
else:
    display(lk_only[top_publication_columns].head(25))

## 7. Type, source, field, and topic analysis

In [ ]:
def frequency_table(data: pd.DataFrame, column: str, top_n: int = 20) -> pd.DataFrame:
    if column not in data.columns:
        return pd.DataFrame({"message": [f"Column not found: {column}"]})
    table = (
        data[column]
        .replace("", np.nan)
        .dropna()
        .astype(str)
        .value_counts()
        .head(top_n)
        .rename_axis(column)
        .reset_index(name="publications")
    )
    table["share_pct"] = (table["publications"] / len(data) * 100).round(2)
    return table


for column in ["type", "source_type", "source_name", "primary_domain", "primary_field", "primary_subfield", "primary_topic"]:
    print(f"\nTop {column}")
    display(frequency_table(lk_only, column, top_n=15))

## 8. Sri Lankan institutions and authors

In [ ]:
def explode_multi_value_column(data: pd.DataFrame, column: str) -> pd.DataFrame:
    if column not in data.columns:
        return pd.DataFrame(columns=[column])
    exploded = data[[column]].copy()
    exploded[column] = exploded[column].apply(split_multi_value)
    exploded = exploded.explode(column)
    exploded[column] = exploded[column].astype(str).str.strip()
    return exploded.loc[exploded[column].ne("")]


def top_multi_value(data: pd.DataFrame, column: str, top_n: int = 25) -> pd.DataFrame:
    exploded = explode_multi_value_column(data, column)
    if exploded.empty:
        return pd.DataFrame({"message": [f"No values available for {column}"]})
    return (
        exploded[column]
        .value_counts()
        .head(top_n)
        .rename_axis(column)
        .reset_index(name="publications")
    )


display(top_multi_value(lk_only, "sri_lankan_institutions", top_n=30))
display(top_multi_value(lk_only, "sri_lankan_authors", top_n=30))

## 9. Open access and language

In [ ]:
for column in ["is_oa", "oa_status", "language"]:
    print(f"\n{column}")
    display(frequency_table(lk_only, column, top_n=20))

## 10. Study-ready charts and graphs

This section creates a complete visual study pack for the strict Sri Lanka-only OpenAlex dataset. Charts are displayed in the notebook and saved as `.png` files under `OUTPUT_DIR / "charts"`.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

CHART_DIR = OUTPUT_DIR / "charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")


def save_chart(fig, filename: str):
    path = CHART_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches="tight")
    print(f"Saved chart: {path}")
    plt.show()
    plt.close(fig)


def clean_series(data: pd.DataFrame, column: str) -> pd.Series:
    if column not in data.columns:
        return pd.Series(dtype="object")
    series = data[column].replace("", np.nan).dropna()
    return series.astype(str).str.strip()


def barh_table(table: pd.DataFrame, label_col: str, value_col: str, title: str, filename: str, color="#2a6f97"):
    if table.empty or label_col not in table.columns or value_col not in table.columns:
        print(f"Skipped chart because table is empty: {title}")
        return
    plot_data = table[[label_col, value_col]].dropna().tail(25).sort_values(value_col)
    if plot_data.empty:
        print(f"Skipped chart because table has no plottable values: {title}")
        return
    height = max(4, min(12, 0.38 * len(plot_data) + 1.5))
    fig, ax = plt.subplots(figsize=(10, height))
    ax.barh(plot_data[label_col], plot_data[value_col], color=color)
    ax.set_title(title)
    ax.set_xlabel(value_col.replace("_", " ").title())
    ax.set_ylabel("")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    save_chart(fig, filename)


def value_counts_table(data: pd.DataFrame, column: str, top_n: int = 20) -> pd.DataFrame:
    if column not in data.columns:
        return pd.DataFrame(columns=[column, "publications", "share_pct"])
    table = frequency_table(data, column, top_n=top_n)
    if "message" in table.columns:
        return pd.DataFrame(columns=[column, "publications", "share_pct"])
    return table


print(f"Charts will be saved to: {CHART_DIR}")

### 10.1 Dashboard metrics

Use these numbers as the headline summary of the filtered dataset.

In [ ]:
metric_columns = ["publications", "total_citations", "mean_citations", "median_citations", "year_min", "year_max"]
metric_table = pd.DataFrame([summary])[metric_columns].T.rename(columns={0: "value"})
display(metric_table)

if "cited_by_count" in lk_only.columns:
    citation_breakdown = pd.DataFrame(
        {
            "metric": ["uncited_publications", "cited_publications", "publications_with_10_plus_citations", "publications_with_100_plus_citations"],
            "count": [
                int((lk_only["cited_by_count"].fillna(0) == 0).sum()),
                int((lk_only["cited_by_count"].fillna(0) > 0).sum()),
                int((lk_only["cited_by_count"].fillna(0) >= 10).sum()),
                int((lk_only["cited_by_count"].fillna(0) >= 100).sum()),
            ],
        }
    )
    citation_breakdown["share_pct"] = (citation_breakdown["count"] / len(lk_only) * 100).round(2)
    display(citation_breakdown)

### 10.2 Publication growth over time

In [ ]:
if "yearly" in globals() and not yearly.empty:
    chart_yearly = yearly.copy()
    chart_yearly["rolling_5yr_publications"] = chart_yearly["publications"].rolling(5, min_periods=1).mean()

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(chart_yearly["publication_year"], chart_yearly["publications"], marker="o", linewidth=1.8, label="Annual publications")
    ax.plot(chart_yearly["publication_year"], chart_yearly["rolling_5yr_publications"], linewidth=2.4, label="5-year rolling average")
    ax.set_title("Sri Lanka-only OpenAlex publications by year")
    ax.set_xlabel("Publication year")
    ax.set_ylabel("Publications")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.legend()
    save_chart(fig, "publications_by_year.png")

    if "total_citations" in chart_yearly.columns:
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.bar(chart_yearly["publication_year"], chart_yearly["total_citations"], color="#5f8d4e")
        ax.set_title("Total citations by publication year")
        ax.set_xlabel("Publication year")
        ax.set_ylabel("Total citations")
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        save_chart(fig, "citations_by_publication_year.png")
else:
    print("Yearly chart skipped because publication_year is unavailable.")

### 10.3 Publication types, sources, languages, and open access

In [ ]:
for column, title, filename, color in [
    ("type", "Publication types", "publication_types.png", "#2a6f97"),
    ("source_type", "Source types", "source_types.png", "#607d8b"),
    ("language", "Publication languages", "languages.png", "#7b2cbf"),
    ("oa_status", "Open access status", "open_access_status.png", "#00876c"),
]:
    table = value_counts_table(lk_only, column, top_n=20)
    display(table)
    barh_table(table, column, "publications", title, filename)

if "is_oa" in lk_only.columns:
    oa_counts = value_counts_table(lk_only.assign(is_oa=lk_only["is_oa"].astype(str)), "is_oa", top_n=5)
    display(oa_counts)

### 10.4 Research domains, fields, subfields, and topics

In [ ]:
for column, title, filename, color in [
    ("primary_domain", "Top OpenAlex domains", "top_domains.png", "#2a9d8f"),
    ("primary_field", "Top OpenAlex fields", "top_fields.png", "#33658a"),
    ("primary_subfield", "Top OpenAlex subfields", "top_subfields.png", "#f26419"),
    ("primary_topic", "Top OpenAlex topics", "top_topics.png", "#8d99ae"),
]:
    table = value_counts_table(lk_only, column, top_n=25)
    display(table.head(25))
    barh_table(table, column, "publications", title, filename, color=color)

if "publication_year" in lk_only.columns and "primary_field" in lk_only.columns:
    top_field_names = value_counts_table(lk_only, "primary_field", top_n=12)["primary_field"].tolist()
    field_year = lk_only.loc[lk_only["primary_field"].isin(top_field_names)].copy()
    field_year = field_year.dropna(subset=["publication_year"])
    if not field_year.empty:
        field_year["publication_year"] = field_year["publication_year"].astype(int)
        pivot = pd.pivot_table(
            field_year,
            index="primary_field",
            columns="publication_year",
            values="title",
            aggfunc="size",
            fill_value=0,
        )
        fig, ax = plt.subplots(figsize=(14, max(5, 0.45 * len(pivot))))
        image = ax.imshow(pivot.values, aspect="auto", cmap="YlGnBu")
        ax.set_title("Publication heatmap: top fields by year")
        ax.set_xlabel("Publication year")
        ax.set_ylabel("")
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns, rotation=90)
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        fig.colorbar(image, ax=ax, label="Publications")
        save_chart(fig, "field_year_heatmap.png")

### 10.5 Institutions, authors, journals, and publishers

In [ ]:
entity_specs = [
    ("sri_lankan_institutions", "Top Sri Lankan institutions", "top_sri_lankan_institutions.png", "#005f73"),
    ("sri_lankan_authors", "Top Sri Lankan authors", "top_sri_lankan_authors.png", "#9b2226"),
]

for column, title, filename, color in entity_specs:
    table = top_multi_value(lk_only, column, top_n=30)
    display(table)
    if "message" not in table.columns:
        barh_table(table, column, "publications", title, filename, color=color)

for column, title, filename, color in [
    ("source_name", "Top journals / sources", "top_sources.png", "#3a5a40"),
    ("publisher", "Top publishers", "top_publishers.png", "#6d597a"),
]:
    table = value_counts_table(lk_only, column, top_n=30)
    display(table)
    barh_table(table, column, "publications", title, filename, color=color)

### 10.6 Citation distribution and high-impact groups

In [ ]:
if "cited_by_count" in lk_only.columns:
    citation_data = lk_only["cited_by_count"].fillna(0).clip(lower=0)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(np.log1p(citation_data), bins=40, color="#4c78a8", edgecolor="white")
    ax.set_title("Citation distribution, log scale")
    ax.set_xlabel("log(1 + cited_by_count)")
    ax.set_ylabel("Publications")
    save_chart(fig, "citation_distribution_log.png")

    citation_bins = pd.cut(
        citation_data,
        bins=[-0.1, 0, 5, 10, 25, 50, 100, np.inf],
        labels=["0", "1-5", "6-10", "11-25", "26-50", "51-100", "100+"],
    )
    citation_bin_table = citation_bins.value_counts().sort_index().rename_axis("citation_range").reset_index(name="publications")
    citation_bin_table["share_pct"] = (citation_bin_table["publications"] / len(lk_only) * 100).round(2)
    display(citation_bin_table)
    barh_table(citation_bin_table, "citation_range", "publications", "Publications by citation range", "citation_ranges.png", color="#bc6c25")

    if "primary_field" in lk_only.columns:
        top_fields_for_citations = value_counts_table(lk_only, "primary_field", top_n=12)["primary_field"].tolist()
        citation_by_field = (
            lk_only.loc[lk_only["primary_field"].isin(top_fields_for_citations)]
            .groupby("primary_field", as_index=False)
            .agg(publications=("title", "size"), total_citations=("cited_by_count", "sum"), mean_citations=("cited_by_count", "mean"), median_citations=("cited_by_count", "median"))
            .sort_values("total_citations", ascending=False)
        )
        citation_by_field[["mean_citations", "median_citations"]] = citation_by_field[["mean_citations", "median_citations"]].round(2)
        display(citation_by_field)
        barh_table(citation_by_field.sort_values("total_citations", ascending=False).head(20), "primary_field", "total_citations", "Total citations by field", "citations_by_field.png", color="#386641")
else:
    print("Citation charts skipped because cited_by_count is unavailable.")

### 10.7 Open access over time

In [ ]:
if "publication_year" in lk_only.columns and "oa_status" in lk_only.columns:
    oa_year = lk_only.dropna(subset=["publication_year"]).copy()
    oa_year["publication_year"] = oa_year["publication_year"].astype(int)
    oa_year["oa_status"] = oa_year["oa_status"].fillna("unknown").astype(str)
    oa_pivot = pd.pivot_table(
        oa_year,
        index="publication_year",
        columns="oa_status",
        values="title",
        aggfunc="size",
        fill_value=0,
    ).sort_index()
    display(oa_pivot.tail(20))

    fig, ax = plt.subplots(figsize=(12, 5))
    oa_pivot.plot(kind="area", stacked=True, ax=ax, alpha=0.86)
    ax.set_title("Open access status over time")
    ax.set_xlabel("Publication year")
    ax.set_ylabel("Publications")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.legend(title="OA status", loc="upper left", bbox_to_anchor=(1, 1))
    save_chart(fig, "open_access_over_time.png")
else:
    print("Open access trend skipped because publication_year or oa_status is unavailable.")

### 10.8 Keywords, concepts, funders, and SDGs

In [ ]:
for column, title, filename, color in [
    ("keywords", "Top keywords", "top_keywords.png", "#4361ee"),
    ("concepts", "Top concepts", "top_concepts.png", "#7209b7"),
    ("funders", "Top funders", "top_funders.png", "#588157"),
    ("sdgs", "Top Sustainable Development Goals", "top_sdgs.png", "#e76f51"),
]:
    table = top_multi_value(lk_only, column, top_n=30)
    display(table)
    if "message" not in table.columns:
        barh_table(table, column, "publications", title, filename, color=color)

### 10.9 Collaboration size and authorship patterns

In [ ]:
if "author_count" in lk_only.columns:
    author_count_data = lk_only["author_count"].dropna().clip(lower=0)
    if not author_count_data.empty:
        fig, ax = plt.subplots(figsize=(10, 5))
        max_authors = int(min(author_count_data.max(), 40))
        ax.hist(author_count_data.clip(upper=max_authors), bins=range(0, max_authors + 2), color="#577590", edgecolor="white")
        ax.set_title("Authors per publication")
        ax.set_xlabel(f"Author count, capped at {max_authors}")
        ax.set_ylabel("Publications")
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        save_chart(fig, "authors_per_publication.png")

        author_count_summary = author_count_data.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).to_frame("author_count")
        display(author_count_summary)
else:
    print("Authorship chart skipped because author_count is unavailable.")

if "institutions_count" in lk_only.columns:
    inst_count = pd.to_numeric(lk_only["institutions_count"], errors="coerce").dropna().clip(lower=0)
    if not inst_count.empty:
        institution_count_table = inst_count.value_counts().sort_index().rename_axis("institutions_count").reset_index(name="publications")
        display(institution_count_table.head(30))
        barh_table(institution_count_table.head(30), "institutions_count", "publications", "Institutions per publication", "institutions_per_publication.png", color="#6a994e")

### 10.10 Recent output and growth

This table helps identify what is growing in the latest available years of the dataset.

In [ ]:
if "publication_year" in lk_only.columns:
    years_available = sorted(int(year) for year in lk_only["publication_year"].dropna().unique())
    recent_years = years_available[-5:]
    older_years = years_available[-10:-5]
    print(f"Recent years used: {recent_years}")
    print(f"Previous comparison years used: {older_years}")

    if recent_years:
        recent_data = lk_only[lk_only["publication_year"].astype("Int64").isin(recent_years)]
        display(frequency_table(recent_data, "primary_field", top_n=20))
        display(frequency_table(recent_data, "source_name", top_n=20))

    if recent_years and older_years and "primary_field" in lk_only.columns:
        recent_counts = lk_only[lk_only["publication_year"].astype("Int64").isin(recent_years)]["primary_field"].value_counts()
        older_counts = lk_only[lk_only["publication_year"].astype("Int64").isin(older_years)]["primary_field"].value_counts()
        growth = pd.concat([older_counts.rename("previous_5_years"), recent_counts.rename("recent_5_years")], axis=1).fillna(0).astype(int)
        growth["change"] = growth["recent_5_years"] - growth["previous_5_years"]
        growth["growth_pct"] = np.where(growth["previous_5_years"] > 0, (growth["change"] / growth["previous_5_years"] * 100).round(2), np.nan)
        growth = growth.sort_values(["change", "recent_5_years"], ascending=False).reset_index(names="primary_field")
        display(growth.head(25))
        barh_table(growth.head(20), "primary_field", "change", "Fastest-growing fields by publication count", "field_growth_recent_5_years.png", color="#0077b6")

### 10.11 Study notes to write your analysis

Use these prompts after running the notebook:

- Which years show the biggest increase in Sri Lanka-only publications?
- Which fields dominate total output, and which fields dominate citations?
- Are the most productive sources also the most cited sources?
- Which institutions and authors appear most often?
- What share of the dataset is open access?
- Which keywords, concepts, funders, and SDGs describe the dataset best?
- Are recent years shifting toward different fields or topics?

## 11. Save analysis tables

In [ ]:
tables = {
    "filter_summary": summary_filter,
    "quality_summary": pd.DataFrame(quality_rows) if "quality_rows" in globals() else pd.DataFrame(),
    "dashboard_metrics": metric_table if "metric_table" in globals() else pd.DataFrame(),
    "citation_breakdown": citation_breakdown if "citation_breakdown" in globals() else pd.DataFrame(),
    "yearly_summary": yearly if "yearly" in globals() else pd.DataFrame(),
    "citation_ranges": citation_bin_table if "citation_bin_table" in globals() else pd.DataFrame(),
    "citations_by_field": citation_by_field if "citation_by_field" in globals() else pd.DataFrame(),
    "open_access_by_year": oa_pivot.reset_index() if "oa_pivot" in globals() else pd.DataFrame(),
    "recent_field_growth": growth if "growth" in globals() else pd.DataFrame(),
    "top_institutions": top_multi_value(lk_only, "sri_lankan_institutions", top_n=100),
    "top_authors": top_multi_value(lk_only, "sri_lankan_authors", top_n=100),
    "top_sources": frequency_table(lk_only, "source_name", top_n=100),
    "top_publishers": frequency_table(lk_only, "publisher", top_n=100),
    "top_domains": frequency_table(lk_only, "primary_domain", top_n=100),
    "top_fields": frequency_table(lk_only, "primary_field", top_n=100),
    "top_subfields": frequency_table(lk_only, "primary_subfield", top_n=100),
    "top_topics": frequency_table(lk_only, "primary_topic", top_n=100),
    "top_keywords": top_multi_value(lk_only, "keywords", top_n=100),
    "top_concepts": top_multi_value(lk_only, "concepts", top_n=100),
    "top_funders": top_multi_value(lk_only, "funders", top_n=100),
    "top_sdgs": top_multi_value(lk_only, "sdgs", top_n=100),
}

for name, table in tables.items():
    path = OUTPUT_DIR / f"{name}.csv"
    table.to_csv(path, index=False)
    print(f"Saved {path}")

print(f"Saved chart images in: {CHART_DIR if 'CHART_DIR' in globals() else OUTPUT_DIR}")